In [ ]:
# --- Standard library: file paths and reproducibility ---
import os
import random
from pathlib import Path

# --- Data handling: arrays and tables ---
import numpy as np
import pandas as pd

# --- Plots: MFCC images, training curves, confusion matrix ---
import matplotlib.pyplot as plt

# --- Audio: load WAV, resample to 16 kHz, compute MFCC/MFE, save clips ---
import librosa
import soundfile as sf

# --- Neural network: layers, training, TFLite int8 export ---
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
import matplotlib.pyplot as plt

# --- Evaluation: per-class metrics and confusion matrix ---
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

# --- Reproducibility: same results on every run ---
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# --- Paths ---
DATA_DIR = Path("data")
RESULTS_DIR = Path("results")
EXPORT_DIR = Path("exported")

# --- Audio and features (same values as in Edge Impulse) ---
SAMPLE_RATE = 16000      # Hz, standard for speech
CLIP_SECONDS = 1.0       # one command fits in 1 s
FRAME_LENGTH = 0.02      # 20 ms frames
FRAME_STRIDE = 0.01      # 10 ms step
N_MFCC = 13              # coefficients per frame
N_MELS = 32              # mel filters

# --- Classes: order defines the network's output indices ---
CLASSES = ["telegram", "elly", "slack", "focus", "noise", "unknown"]

# --- Training ---
EPOCHS = 100
LEARNING_RATE = 0.005
BATCH_SIZE = 32

# --- Check that everything is installed ---
print("TensorFlow:", tf.__version__)
print("librosa:", librosa.__version__)

In [ ]:
# --- Parse file name: label.speaker.number.wav -> label, speaker ---
def parse_name(path):
    parts = path.stem.split(".")
    if len(parts) < 3:
        return None
    return {"path": path, "label": parts[0].lower(), "speaker": parts[1].lower()}

# --- Build the dataset index: one row per clip ---
all_wavs = sorted(DATA_DIR.rglob("*.wav"))
rows = [r for p in all_wavs if (r := parse_name(p))]
skipped = [p.name for p in all_wavs if parse_name(p) is None]
df = pd.DataFrame(rows, columns=["path", "label", "speaker"])

# --- Keep only the classes we train on ---
ignored = sorted(set(df["label"]) - set(CLASSES))
df = df[df["label"].isin(CLASSES)].reset_index(drop=True)

# --- Summary: how many clips per class and per speaker ---
print(f"Clips: {len(df)}")
if skipped:
    print("Skipped, wrong file name:", skipped[:10])
if ignored:
    print("Ignored labels:", ignored)
if df.empty:
    print(f"No clips found in {DATA_DIR.resolve()}")
else:
    display(pd.crosstab(df["label"], df["speaker"], margins=True))

In [ ]:
# --- Target length: 1 s at 16 kHz = 16000 samples ---
N_SAMPLES = int(SAMPLE_RATE * CLIP_SECONDS)

# --- Load one clip: resample to 16 kHz, mono, exactly 1 s ---
def load_clip(path):
    y, _ = librosa.load(path, sr=SAMPLE_RATE, mono=True)
    if len(y) > N_SAMPLES:
        # longer than 1 s: keep the loudest 1 s window (where the word is)
        cs = np.concatenate([[0], np.cumsum(y ** 2)])
        energy = cs[N_SAMPLES:] - cs[:-N_SAMPLES]
        start = int(np.argmax(energy))
        y = y[start:start + N_SAMPLES]
    else:
        # shorter than 1 s: pad with silence at the end
        y = np.pad(y, (0, N_SAMPLES - len(y)))
    return y.astype(np.float32)

# --- Load all clips into one array: (number of clips, 16000) ---
if len(df):
    X_audio = np.stack([load_clip(p) for p in df["path"]])
else:
    X_audio = np.empty((0, N_SAMPLES), dtype=np.float32)
print("Audio array:", X_audio.shape)

# --- Quick check: waveform of the first clip ---
if len(df):
    plt.figure(figsize=(10, 2.5))
    plt.plot(np.arange(N_SAMPLES) / SAMPLE_RATE, X_audio[0])
    plt.title(f'{df.loc[0, "label"]} ({df.loc[0, "speaker"]})')
    plt.xlabel("time, s")
    plt.tight_layout()
    plt.show()

In [ ]:
WIN = int(SAMPLE_RATE * FRAME_LENGTH)   # 320 samples
HOP = int(SAMPLE_RATE * FRAME_STRIDE)   # 160 samples
N_FFT = 512

def compute_mfcc(y):
    m = librosa.feature.mfcc(y=y, sr=SAMPLE_RATE, n_mfcc=N_MFCC, n_mels=N_MELS,
                             n_fft=N_FFT, win_length=WIN, hop_length=HOP,
                             window="hann", center=False)
    return m.T.astype(np.float32)       # (frames, 13)

In [ ]:
class_to_idx = {c: i for i, c in enumerate(CLASSES)}
X = np.stack([compute_mfcc(load_clip(p)) for p in df["path"]])   # (N, frames, 13)
y = df["label"].map(class_to_idx).values.astype(np.int64)
print(X.shape, y.shape)

In [ ]:
HELD_OUT_SPEAKER = df["speaker"].unique()[0]     # change by hand to choose who is the test speaker
test_mask = (df["speaker"] == HELD_OUT_SPEAKER).values
X_trainval, y_trainval = X[~test_mask], y[~test_mask]
X_test, y_test = X[test_mask], y[test_mask]
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.15, stratify=y_trainval, random_state=SEED)
print(len(X_train), len(X_val), len(X_test))